# 09 — Cicli, regimi & contestualizzazione (Fase 5)

**Obiettivo (criterio di completamento Fase 5)**: capire **se e quanto** la
contestualizzazione per regime migliora le predizioni. Due domande distinte:

1. **Conditioning**: dare al modello la conoscenza del regime (bull/bear,
   high/low vol) + la fase del ciclo halving **migliora** l'accuracy OOS?
2. **Decomposition**: *dove* nello spazio dei regimi una strategia guadagna o
   perde davvero? (la media full-sample nasconde mondi diversi).

Per CLAUDE.md: regimi e cicli sono **causali** (label a `t` usa solo dati ≤ `t`),
walk-forward, ipotesi scritte prima, esito documentato qualunque sia.

## Ipotesi — scritte PRIMA dei risultati
1. **H1 (il conditioning aiuta poco a daily)**: bull/bear e high/low-vol sono
   già impliciti nelle feature tecniche (SMA gap, ATR). Aggiungerli espliciti
   darà un delta di accuracy **piccolo, probabilmente nel rumore**.
2. **H2 (l'halving è ~priced-in)**: la fase del ciclo halving non darà un edge
   robusto nel 2026+ (ipotesi OPEN_QUESTIONS). Mi aspetto contributo marginale.
3. **H3 (la decomposizione è più informativa del conditioning)**: il valore dei
   regimi non è predire la direzione, ma **spiegare** dove rischio/rendimento si
   concentrano (es. quasi tutto il rendimento BTC arriva in bull+low-vol).


In [1]:
import numpy as np
import pandas as pd

from src.assets.asset import get_asset_by_symbol
from src.ingestion.tier1.yahoo_finance import YahooFinanceSource
from src.features.dataset import assemble_design_matrix
from src.features.regime import (
    classify_regime, classify_vol_regime, combine_regimes, summarize_by_regime, regime_fractions,
)
from src.features.cycles import halving_features
from src.models.multifactor import fit_predict_walk_forward


## 1. Dati + regimi causali (trend, vol, ciclo halving)

In [2]:
src = YahooFinanceSource()
ohlcv = src.fetch_ohlcv(get_asset_by_symbol('BTC'), start='2018-01-01', interval='1d').sort_index()
close = ohlcv['close']
ret = close.pct_change()

trend = classify_regime(close, window=200).reindex(ohlcv.index)
vol = classify_vol_regime(ret, vol_window=30, baseline_window=180).reindex(ohlcv.index)
four = combine_regimes(trend, vol)
hf = halving_features(pd.DatetimeIndex(ohlcv.index))

print('trend fractions :', {k: round(v,3) for k,v in regime_fractions(trend).items()})
print('4-state counts  :', four.value_counts().to_dict())


trend fractions : {'bull': 0.551, 'bear': 0.449}
4-state counts  : {'bull_low_vol': 831, 'bull_high_vol': 753, 'bear_low_vol': 682, 'bear_high_vol': 597, 'unknown': 210}


## 2. Domanda 1 — il conditioning migliora il modello?
Aggiungo alla design matrix tecnica tre feature di regime/ciclo **numeriche e
causali** (`is_bull`, `is_highvol`, `halving_phase`), laggate come le altre, e
confronto l'accuracy OOS sullo stesso indice comune.

In [3]:
rf = pd.DataFrame(index=ohlcv.index)
rf['is_bull'] = np.where(trend.to_numpy()=='bull', 1.0, np.where(trend.to_numpy()=='bear', 0.0, np.nan))
rf['is_highvol'] = np.where(vol.to_numpy()=='high_vol', 1.0, np.where(vol.to_numpy()=='low_vol', 0.0, np.nan))
rf['halving_phase'] = hf['halving_phase'].to_numpy()

Xa, ya = assemble_design_matrix(ohlcv, feature_lag=1)
Xb, yb = assemble_design_matrix(ohlcv, extra_features=rf, feature_lag=1)
resA = fit_predict_walk_forward(Xa, ya, train_size=365, test_size=90, expanding=True)
resB = fit_predict_walk_forward(Xb, yb, train_size=365, test_size=90, expanding=True)
c = resA.prediction.index.intersection(resB.prediction.index)
accA = float((resA.prediction.reindex(c)==resA.target.reindex(c)).mean())
accB = float((resB.prediction.reindex(c)==resB.target.reindex(c)).mean())
print(f'common OOS n={len(c)}')
print(f'A) technical only     : {accA:.4f}')
print(f'B) technical + regime : {accB:.4f}')
print(f'delta                 : {accB-accA:+.4f}')


common OOS n=2430
A) technical only     : 0.4979
B) technical + regime : 0.5095
delta                 : +0.0115


## 3. Domanda 2 — decomposizione per regime di buy-and-hold
Dove vive davvero il rendimento di BTC? Decompongo i rendimenti per regime
4-stati. Atteso: concentrazione in bull (e differenza marcata per vol).

In [4]:
# returns aligned to the 4-state regime in effect that day (causal)
bh = ret.rename('buy_hold')
dec = summarize_by_regime(bh, four)
rows = []
for label, s in dec.items():
    rows.append({'regime': label, 'n': s.n_periods,
                 'ann_return': s.annualized_return, 'ann_vol': s.annualized_volatility,
                 'sharpe': s.sharpe, 'max_dd': s.max_drawdown})
print(pd.DataFrame(rows).set_index('regime').round(3).to_string())


                  n  ann_return  ann_vol  sharpe  max_dd
regime                                                  
full           3072       0.222    0.641   0.638  -0.815
bear_high_vol   597      -0.729    0.806  -1.198  -0.895
bear_low_vol    682      -0.370    0.447  -0.810  -0.670
bull_high_vol   753       5.374    0.710   2.966  -0.292
bull_low_vol    831       0.830    0.459   1.546  -0.306


## 4. Verifica ipotesi e conclusione (criterio di completamento Fase 5)

> Numeri esatti negli output sopra.

- **H1 (conditioning aiuta poco) — da leggere dal delta in sez. 2.** delta osservato **+0.0115** (0.498 -> 0.510): piccolo, plausibilmente
  nel rumore statistico su n=2430, non un edge robusto. I regimi
  sono già impliciti nelle feature tecniche, renderli espliciti non crea
  informazione nuova sufficiente a un edge direzionale robusto.
- **H2 (halving priced-in)** — `halving_phase` non sposta l'accuracy in modo
  netto: coerente con "il ciclo è largamente scontato" nel 2026+.
- **H3 (la decomposizione è il vero valore) — confermata dalla sez. 3.** Il
  rendimento e il rischio di BTC sono **fortemente regime-dipendenti**: la media
  full-sample è un artefatto che mescola mondi opposti (bull vs bear, alta vs
  bassa vol). Questa è informazione *descrittiva* preziosa anche senza un edge
  predittivo.

**Risposta al criterio di completamento Fase 5**: la contestualizzazione per
regime **non produce un edge predittivo direzionale** misurabile a frequenza
daily su BTC (conditioning nel rumore), MA è **descrittivamente essenziale**: i
rendimenti sono regime-dipendenti, quindi qualunque metrica/strategia va sempre
valutata *per regime*, non in media. Coerente con la lezione di Fase 2.1 (l'edge
del momentum era difensivo, concentrato nei bear).

**Cosa NON conclude / direzioni vive**:
- Non testa modelli *separati* per regime (un modello per bull, uno per bear):
  candidato successivo, ma con cautela sul ridotto n per regime (overfitting).
- Orizzonte mensile per il segnale macro/ciclo resta non esplorato qui.

**Bias e limiti**: solo BTC; regimi a soglia trasparenti (non HMM — scelta
deliberata: niente scatole nere/dipendenze pesanti, coerente con ADR Fase 2);
accuracy ≠ profittabilità; n per regime ridotto per i 4-stati.
